In [1]:
from collections import defaultdict
import random
import numpy as np
import pandas as pd
import json
import pickle
import gzip
import tqdm

In [2]:
# K-core user_core item_core
def check_Kcore(user_items, user_core, item_core):
    user_count = defaultdict(int)
    item_count = defaultdict(int)
    for user, items in user_items.items():
        for item in items:
            user_count[user] += 1
            item_count[item] += 1

    for user, num in user_count.items():
        if num < user_core:
            return user_count, item_count, False
    for item, num in item_count.items():
        if num < item_core:
            return user_count, item_count, False
    return user_count, item_count, True # 已经保证Kcore

In [ ]:
#  K-core
def filter_Kcore(user_items, user_core, item_core): # user items
    user_count, item_count, isKcore = check_Kcore(user_items, user_core, item_core)
    while not isKcore:
        for user, num in user_count.items():
            if user_count[user] < user_core:  
                user_items.pop(user)
            else:
                for item in user_items[user]:
                    if item_count[item] < item_core:
                        user_items[user].remove(item)
        user_count, item_count, isKcore = check_Kcore(user_items, user_core, item_core)
    return user_items


In [4]:
def add_comma(num):
    # 1000000 -> 1,000,000
    str_num = str(num)
    res_num = ''
    for i in range(len(str_num)):
        res_num += str_num[i]
        if (len(str_num)-i-1) % 3 == 0:
            res_num += ','
    return res_num[:-1]

In [13]:
def id_map(user_items): # user_items dict

    user2id = {} # raw 2 uid
    item2id = {} # raw 2 iid
    id2user = {} # uid 2 raw
    id2item = {} # iid 2 raw
    user_id = 1
    item_id = 1
    final_data = {}
    for user, items in user_items.items():
        if user not in user2id:
            user2id[user] = str(user_id)
            id2user[str(user_id)] = user
            user_id += 1
        iids = [] # item id lists
        for item in items:
            if item not in item2id:
                item2id[item] = str(item_id)
                id2item[str(item_id)] = item
                item_id += 1
            iids.append(item2id[item])
        uid = user2id[user]
        final_data[uid] = iids
    data_maps = {
        'user2id': user2id,
        'item2id': item2id,
        'id2user': id2user,
        'id2item': id2item
    }
    return final_data, user_id-1, item_id-1, data_maps

In [ ]:
def LastFM():
    user_core = 5
    item_core = 5
    datas = []
    data_file = '/home/v100/workspace/lwx/HM4SR/dataset/lastfm/user_taggedartists-timestamps.dat'
    lines = open(data_file).readlines()
    for line in tqdm.tqdm(lines[1:]):
        user, item, attribute, timestamp = line.strip().split('\t')
        datas.append((user, item, int(timestamp)))

   
    
    # 
    user_seq = {}
    user_seq_notime = {}
    for data in datas:
        user, item, time = data
        if user in user_seq:
            if item not in user_seq_notime[user]:
                user_seq[user].append((item, time))
                user_seq_notime[user].append(item)
            else:
                continue
        else:
            user_seq[user] = []
            user_seq_notime[user] = []

            user_seq[user].append((item, time))
            user_seq_notime[user].append(item)
    
    triplet_list = [
    (user_id, item_id, timestamp) 
    for user_id, interactions in user_seq.items() 
    for (item_id, timestamp) in interactions
    ]
    df = pd.DataFrame(triplet_list, columns=['user_id', 'item_id', 'timestamp'])
    df.columns = ['user_id:token', 'item_id:token', 'timestamp:float']
    output_file_path = 'lastfm.inter'  # 
    df.to_csv(output_file_path, sep='\t', index=False)
    for user, item_time in user_seq.items():
        item_time.sort(key=lambda x: x[1])  
        items = []
        for t in item_time:
            items.append(t[0])
        user_seq[user] = items

    user_items = filter_Kcore(user_seq, user_core=user_core, item_core=item_core)
    print(f'User {user_core}-core complete! Item {item_core}-core complete!')

    user_items, user_num, item_num, data_maps = id_map(user_items)  # new_num_id
    user_count, item_count, _ = check_Kcore(user_items, user_core=user_core, item_core=item_core)
    user_count_list = list(user_count.values())
    user_avg, user_min, user_max = np.mean(user_count_list), np.min(user_count_list), np.max(user_count_list)
    item_count_list = list(item_count.values())
    item_avg, item_min, item_max = np.mean(item_count_list), np.min(item_count_list), np.max(item_count_list)
    interact_num = np.sum([x for x in user_count_list])
    sparsity = (1 - interact_num / (user_num * item_num)) * 100
    show_info = f'Total User: {user_num}, Avg User: {user_avg:.4f}, Min Len: {user_min}, Max Len: {user_max}\n' + \
                f'Total Item: {item_num}, Avg Item: {item_avg:.4f}, Min Inter: {item_min}, Max Inter: {item_max}\n' + \
                f'Iteraction Num: {interact_num}, Sparsity: {sparsity:.2f}%'
    print(show_info)

    # attribute_file = './data_path/artist2attributes.json'

    # meta_item2attribute = json.loads(open(attribute_file).readline())

    # # 做映射
    # attribute2id = {}
    # id2attribute = {}
    # attribute_id = 1
    # item2attributes = {}
    # attribute_lens = []
    # # load id map
    # for iid, attributes in meta_item2attribute.items():
    #     if iid in list(data_maps['item2id'].keys()):
    #         item_id = data_maps['item2id'][iid]
    #         item2attributes[item_id] = []
    #         for attribute in attributes:
    #             if attribute not in attribute2id:
    #                 attribute2id[attribute] = attribute_id
    #                 id2attribute[attribute_id] = attribute
    #                 attribute_id += 1
    #             item2attributes[item_id].append(attribute2id[attribute])
    #         attribute_lens.append(len(item2attributes[item_id]))
    # print(f'after delete, attribute num:{len(attribute2id)}')
    # print(f'attributes len, Min:{np.min(attribute_lens)}, Max:{np.max(attribute_lens)}, Avg.:{np.mean(attribute_lens):.4f}')
    # # 更新datamap
    # data_maps['attribute2id'] = attribute2id
    # data_maps['id2attribute'] = id2attribute

    # data_name = 'LastFM'
    # print(f'{data_name} & {add_comma(user_num)}& {add_comma(item_num)} & {user_avg:.1f}'
    #       f'& {item_avg:.1f}& {add_comma(interact_num)}& {sparsity:.2f}\%&{add_comma(len(attribute2id))}&'
    #       f'{np.mean(attribute_lens):.1f} \\')

    # # -------------- Save Data ---------------
    # # one user one line
    # data_file = 'data/' + data_name + '.txt'
    # item2attributes_file = 'data/' + data_name + '_item2attributes.json'

    # with open(data_file, 'w') as out:
    #     for user, items in user_items.items():
    #         out.write(user + ' ' + ' '.join(items) + '\n')

    # json_str = json.dumps(item2attributes)
    # with open(item2attributes_file, 'w') as out:
    #     out.write(json_str)

In [ ]:
LastFM()

100%|██████████| 186479/186479 [00:01<00:00, 133100.09it/s]


In [ ]:
def local_minmax_day(dataset, data_path):
    with open(dataset + '/' + dataset + '.inter', 'r') as f:
        line = f.readlines()[1:]
    min_date, max_date = 9999999, 0
    for l in line:
        user, _, timestamp = l.strip().split('\t')
        user, timestamp = int(user), int(timestamp)
        min_date = min(int(timestamp / 86400), min_date)
        max_date = max(int(timestamp / 86400), max_date)
    with open(data_path, 'wb') as f:
        print(min_date)
        print(max_date)
        pickle.dump((min_date, max_date), f)

In [ ]:
def local_timestamp(dataset, data_path):
    with open(dataset + '/' + dataset + '.inter', 'r') as f:
        line = f.readlines()[1:]
    tmp = [-1, 0]
    max_interval = 0
    for l in line:
        user, _, timestamp = l.strip().split('\t')
        user, timestamp = int(user), int(timestamp)
        if user != tmp[0]:
            tmp = [user, timestamp]
        else:
            interval = timestamp - tmp[1]
            tmp[1] = timestamp
            max_interval = max(interval, max_interval)
    max_interval = torch.log2(torch.tensor(max_interval) + 1).item()
    with open(data_path, 'wb') as f:
        print(max_interval)
        pickle.dump(max_interval, f)

In [ ]:
#dataset Amazon_Beauty
dataset = 'lastfm'
local_timestamp(dataset, 'interval_num')
local_minmax_day(dataset, 'minmax_num')